# ML-08 Capstone Modeling Lane (Refresh)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/oumaklaus/ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

My lane is Refresh, or Content Opportunity Scoring: rank the pages a reviewer should look at first. Week 4 gave me a hand written rule to beat. This notebook builds the model and puts the two side by side on the same rows, the same folds, and the same metric.

Same slice as ML-04 and ML-07: features from March 2026, the decline label from April 2026, pages with `gsc_data_available IS TRUE`. The label is unchanged, `apr_impr < 0.8 * mar_impr`.

Order of work: method choice, split design, the comparison table, then errors and what the model leans on.

## 1. Method choice and why

**The decision this feeds.** A reviewer works down a queue from the top. So the number that matters is precision at small K, not overall accuracy. A model that is right about the first fifty pages is worth more here than one with a prettier average.

**What I am fitting.** Two models, in order of how much they ask me to trust them:

1. **Logistic Regression.** The honest step up from a hand written rule. It keeps the same shape as my baseline, a weighted sum of readable inputs, but it learns the weights instead of me guessing them. If a linear model already beats the rule, I do not need anything heavier.
2. **Random Forest.** The challenger, to test whether the relationships are actually curved rather than straight. Decline versus position is a good example: both very strong and very weak positions can behave differently from the middle, and a linear model cannot bend to that.

I report both against the rule, plus a stratified dummy as the floor below the floor. Complexity has to earn its place. If the forest does not clearly beat the regression on the metric that matters, I say so and keep the simpler model.

**One new feature family, and why it is fair.** My ML-07 rule leaned on age and volume. Both are static: they describe what a page *is*, not where it is *heading*. So I add within March momentum, the second half of the month over the first half. A page already sliding while March is still running is the clearest decision time hint that April will be worse.

This is not leakage, and the cell below proves it two ways. The halves sum exactly to the March total, so they are built from feature month days only, and April appears nowhere in them.

**What I deliberately leave out**, same as ML-07: `content_updated_date`, `last_optimized_date`, and `optimization_eligible_date` all carry a July snapshot date in this build, which is after my decision date, so they are future information.

In [1]:
# Setup and one pass over the warehouse. Token comes from the environment, Colab Secrets,
# or a prompt. It is NEVER written into this notebook (public repo).
import os, json, duckdb, numpy as np, pandas as pd
pd.set_option("display.width", 200)

def _hf_token():
    tok = os.environ.get("HF_TOKEN")
    if tok:
        return tok
    try:
        from google.colab import userdata
        return userdata.get("HF_TOKEN")
    except Exception:
        import getpass
        return getpass.getpass("HF read token: ")

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{_hf_token()}')")

REL  = "hf://datasets/FlyRank/internship-warehouse"
FACT = f"{REL}/fact_content_daily_performance"
MAR  = f"read_parquet('{FACT}/month=2026-03/*.parquet')"   # features
APR  = f"read_parquet('{FACT}/month=2026-04/*.parquet')"   # label window
DIMC = f"read_parquet('{REL}/dim_content.parquet')"        # age + client for grouping

def _out_dir():
    d = os.getcwd()
    for _ in range(6):
        if os.path.isdir(os.path.join(d, ".git")) or os.path.isdir(os.path.join(d, "work", "outputs")):
            return os.path.join(d, "work", "outputs")
        d = os.path.dirname(d)
    return "work/outputs"
OUT = _out_dir(); os.makedirs(OUT, exist_ok=True)
print("connected; outputs ->", OUT)

connected; outputs -> /home/zuko/ml-internship/work/outputs


In [2]:
# One query for everything March: the ML-04 features plus the two half month aggregates
# that momentum is built from. Splitting March at the 15th keeps it inside the feature month.
feat = con.sql(f"""
    SELECT content_hash_id,
           SUM(gsc_impressions)                                      AS mar_impr,
           SUM(gsc_clicks)                                           AS mar_clicks,
           100.0 * SUM(gsc_clicks) / NULLIF(SUM(gsc_impressions), 0) AS ctr,
           SUM(gsc_sum_position)   / NULLIF(SUM(gsc_impressions), 0) AS avg_position,
           COUNT(*) FILTER (WHERE gsc_impressions > 0)               AS days_with_impr,
           SUM(CASE WHEN report_date <= DATE '2026-03-15' THEN gsc_impressions ELSE 0 END) AS h1_impr,
           SUM(CASE WHEN report_date >  DATE '2026-03-15' THEN gsc_impressions ELSE 0 END) AS h2_impr,
           SUM(CASE WHEN report_date <= DATE '2026-03-15' THEN gsc_clicks ELSE 0 END)      AS h1_clicks,
           SUM(CASE WHEN report_date >  DATE '2026-03-15' THEN gsc_clicks ELSE 0 END)      AS h2_clicks
    FROM {MAR}
    WHERE gsc_data_available IS TRUE
    GROUP BY content_hash_id
    HAVING SUM(gsc_impressions) > 0
""").df()

dc = con.sql(f"SELECT content_hash_id, client_hash_id, content_created_date FROM {DIMC}").df()
dc["content_created_date"] = pd.to_datetime(dc["content_created_date"])
m = feat.merge(dc, on="content_hash_id", how="left")
m["age_days"] = (pd.Timestamp("2026-03-31") - m["content_created_date"]).dt.days

lab = con.sql(f"SELECT content_hash_id, SUM(gsc_impressions) AS apr_impr FROM {APR} "
              f"WHERE gsc_data_available IS TRUE GROUP BY content_hash_id").df()
m = m.merge(lab, on="content_hash_id", how="left")
m["apr_impr"] = m["apr_impr"].fillna(0)
m["decline"]  = (m["apr_impr"] < 0.8 * m["mar_impr"]).astype(int)
m = m[m["age_days"].notna()].copy()

# Momentum. The +1 keeps pages that went to zero in the second half from dividing by nothing.
m["mom_impr"]   = (m["h2_impr"] + 1) / (m["h1_impr"] + 1)
m["mom_clicks"] = (m["h2_clicks"] + 1) / (m["h1_clicks"] + 1)

base = m["decline"].mean()
print(f"pages: {len(m):,}   clients: {m['client_hash_id'].nunique()}   base decline rate: {base:.4f}")

# Proof that momentum is decision time safe, not a peek at April.
gap = (m["h1_impr"] + m["h2_impr"] - m["mar_impr"]).abs().max()
print(f"\ncheck: max |h1 + h2 - march total| = {gap:.1f}  (0 means the halves are pure March)")
print("check: April enters only through 'decline'; no half month column reads it.")

print("\nWithin March momentum vs April decline")
mb = pd.cut(m["mom_impr"], [0, 0.5, 0.9, 1.1, 2.0, 1e9],
            labels=["<0.5 halved", "0.5-0.9 sliding", "0.9-1.1 flat", "1.1-2 rising", ">2 spiking"])
print(m.groupby(mb, observed=True)["decline"].agg(decline_rate="mean", n="size").round(3).to_string())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

pages: 176,738   clients: 47   base decline rate: 0.5319

check: max |h1 + h2 - march total| = 0.0  (0 means the halves are pure March)
check: April enters only through 'decline'; no half month column reads it.

Within March momentum vs April decline
                 decline_rate      n
mom_impr                            
<0.5 halved             0.782  26796
0.5-0.9 sliding         0.673  31758
0.9-1.1 flat            0.573  20005
1.1-2 rising            0.475  51275
>2 spiking              0.338  46904


## 2. Split design

**Grouped by client, five folds, no page from a test client ever seen in training.**

The reason is specific to this slice, not a habit. My ML-07 weak picks found that the top 100 of my baseline queue was a single client. When I look at decline rate per client it is nowhere near uniform: among clients with at least 500 pages it runs from about 0.22 to about 0.88, around a base of 0.53. Clients differ that much because a client is a site, with its own template, publishing rhythm, season, and Search Console history.

So a page's client is a strong clue about whether it declines. Under a random split, pages from the same client sit on both sides of the line, and a model can pick up client habits instead of page level decay. It scores well and teaches me nothing about the next client FlyRank onboards.

Grouping by client asks the question I actually care about: does this generalise to a site the model has never seen? That is the harder question and the honest one.

The time direction is already handled by construction, features in March and label in April, so the split only needs to fix the client problem.

The cell below runs the same model both ways so the size of the difference is visible rather than asserted.

In [3]:
from sklearn.model_selection import GroupKFold, train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score

STATIC = ["mar_impr", "mar_clicks", "ctr", "avg_position", "days_with_impr", "age_days"]
MOMENT = ["mom_impr", "mom_clicks", "h1_impr", "h2_impr"]
FEATS  = STATIC + MOMENT

X = m[FEATS].fillna(0.0)
y = m["decline"].values
groups = m["client_hash_id"].values
gkf = GroupKFold(n_splits=5)

print("How much does the split choice change the answer? Same model, same features.\n")
tr, te = next(gkf.split(X, y, groups))
lr_g = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)).fit(X.iloc[tr], y[tr])
auc_g = roc_auc_score(y[te], lr_g.predict_proba(X.iloc[te])[:, 1])

rtr, rte = train_test_split(np.arange(len(m)), test_size=0.2, stratify=y, random_state=42)
lr_r = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)).fit(X.iloc[rtr], y[rtr])
auc_r = roc_auc_score(y[rte], lr_r.predict_proba(X.iloc[rte])[:, 1])

print(f"  random split    AUC {auc_r:.4f}   test clients also in train: {len(set(groups[rte]) & set(groups[rtr]))}")
print(f"  grouped split   AUC {auc_g:.4f}   test clients also in train: {len(set(groups[te]) & set(groups[tr]))}")
print(f"  the random split flatters the model by {auc_r - auc_g:+.4f} AUC")

print("\nDecline rate really does move by client (why grouping matters):")
cl = m.groupby("client_hash_id")["decline"].agg(decline_rate="mean", pages="size")
cl = cl[cl["pages"] >= 500].sort_values("decline_rate")
print(f"  {len(cl)} clients with 500+ pages, decline rate from "
      f"{cl['decline_rate'].min():.3f} to {cl['decline_rate'].max():.3f} (base {base:.3f})")

print("\nFold shape:")
for f, (a, b_) in enumerate(gkf.split(X, y, groups)):
    print(f"  fold {f}: test pages {len(b_):,}  test clients {len(set(groups[b_])):2d}  "
          f"test base rate {y[b_].mean():.3f}")

How much does the split choice change the answer? Same model, same features.



  random split    AUC 0.6436   test clients also in train: 45


  grouped split   AUC 0.5548   test clients also in train: 0
  the random split flatters the model by +0.0889 AUC

Decline rate really does move by client (why grouping matters):
  27 clients with 500+ pages, decline rate from 0.217 to 0.883 (base 0.532)

Fold shape:


  fold 0: test pages 35,348  test clients  8  test base rate 0.437
  fold 1: test pages 35,347  test clients  9  test base rate 0.584
  fold 2: test pages 35,347  test clients  8  test base rate 0.460
  fold 3: test pages 35,348  test clients 11  test base rate 0.554


  fold 4: test pages 35,348  test clients 11  test base rate 0.624


## 3. Train and compare against my baseline

Every scorer below sees the identical five folds and the identical test rows. My ML-07 rule is rescored inside each fold rather than reused from Week 4, which is the only way the comparison is fair.

The five scorers:

* **baseline_w04**, my frozen rule: `age_days` if `mar_impr >= 50` and `age_days >= 90`, else 0.
* **dummy**, stratified guessing, the floor below the floor.
* **logreg_static**, logistic regression on the six static features only. This is the fair test of whether learned weights beat my hand picked thresholds using the same kind of information.
* **logreg_momentum** and **rf_momentum**, the same models once within March momentum is added.

Metrics: precision at K because that is the queue, ROC AUC and PR AUC for the ranking as a whole. I report the mean across folds **and the spread**, because with 47 clients a fold holds only 8 to 11 of them and single fold numbers move a lot.

In [4]:
def baseline_w04(df):
    """My frozen ML-07 rule, rescored on whatever rows it is given."""
    flag = (df["mar_impr"] >= 50) & (df["age_days"] >= 90)
    return np.where(flag, df["age_days"], 0).astype(float)

def p_at_k(scores, yy, k, tiebreak=None):
    """Ties in the rule's score are broken by impressions, exactly as the ML-07 queue did."""
    tb = tiebreak if tiebreak is not None else np.zeros(len(scores))
    order = np.lexsort((-np.asarray(tb), -np.asarray(scores)))
    return np.asarray(yy)[order[:k]].mean()

def evaluate(name, scores, yy, tiebreak=None):
    return {"model": name,
            "AUC": roc_auc_score(yy, scores),
            "PR_AUC": average_precision_score(yy, scores),
            **{f"p@{k}": p_at_k(scores, yy, k, tiebreak) for k in (10, 50, 100, 500)}}

rows, err = [], []
for fold, (tr, te) in enumerate(gkf.split(X, y, groups)):
    Xtr, Xte, ytr, yte = X.iloc[tr], X.iloc[te], y[tr], y[te]
    tb = m["mar_impr"].values[te]

    rows.append({"fold": fold, **evaluate("baseline_w04", baseline_w04(m.iloc[te]), yte, tb)})
    dm = DummyClassifier(strategy="stratified", random_state=0).fit(Xtr, ytr)
    rows.append({"fold": fold, **evaluate("dummy", dm.predict_proba(Xte)[:, 1], yte)})

    lr_s = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)).fit(Xtr[STATIC], ytr)
    rows.append({"fold": fold, **evaluate("logreg_static", lr_s.predict_proba(Xte[STATIC])[:, 1], yte)})

    lr_m = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)).fit(Xtr, ytr)
    p_lr = lr_m.predict_proba(Xte)[:, 1]
    rows.append({"fold": fold, **evaluate("logreg_momentum", p_lr, yte)})

    rf = RandomForestClassifier(n_estimators=200, min_samples_leaf=100,
                                n_jobs=-1, random_state=42).fit(Xtr, ytr)
    p_rf = rf.predict_proba(Xte)[:, 1]
    rows.append({"fold": fold, **evaluate("rf_momentum", p_rf, yte)})

    err.append(pd.DataFrame({"fold": fold, "y": yte, "p_rf": p_rf, "p_lr": p_lr,
                             "mar_impr": m["mar_impr"].values[te],
                             "age_days": m["age_days"].values[te],
                             "mom_impr": m["mom_impr"].values[te]}))
    print(f"  fold {fold} trained")

res = pd.DataFrame(rows)
order = ["dummy", "baseline_w04", "logreg_static", "logreg_momentum", "rf_momentum"]
mean_tbl = res.drop(columns="fold").groupby("model").mean().reindex(order).round(3)
std_tbl  = res.drop(columns="fold").groupby("model").std().reindex(order).round(3)

print(f"\nMODEL VS BASELINE, mean over 5 client grouped folds (base rate {base:.3f})")
print(mean_tbl.to_string())
print("\nSpread across folds (standard deviation). Read this before believing any single number.")
print(std_tbl.to_string())
print("\nPer fold precision@50, so the variance is visible rather than hidden in an average:")
print(res.pivot(index="fold", columns="model", values="p@50").reindex(columns=order).round(2).to_string())

  fold 0 trained


  fold 1 trained


  fold 2 trained


  fold 3 trained


  fold 4 trained

MODEL VS BASELINE, mean over 5 client grouped folds (base rate 0.532)
                   AUC  PR_AUC  p@10   p@50  p@100  p@500
model                                                    
dummy            0.499   0.531  0.70  0.560  0.552  0.539
baseline_w04     0.532   0.579  0.52  0.572  0.614  0.622
logreg_static    0.522   0.554  0.58  0.560  0.586  0.593
logreg_momentum  0.573   0.593  0.82  0.796  0.790  0.743
rf_momentum      0.671   0.672  0.90  0.820  0.790  0.753

Spread across folds (standard deviation). Read this before believing any single number.
                   AUC  PR_AUC   p@10   p@50  p@100  p@500
model                                                     
dummy            0.003   0.081  0.245  0.164  0.126  0.084
baseline_w04     0.132   0.122  0.427  0.400  0.350  0.340
logreg_static    0.092   0.088  0.179  0.086  0.128  0.195
logreg_momentum  0.090   0.095  0.130  0.122  0.115  0.151
rf_momentum      0.064   0.102  0.173  0.148  0.140  0.196

Per

### What the table says

Read against the base rate and against the rule, not against perfection.

**The Week 4 rule is much weaker than Week 4 suggested.** In ML-07 it scored precision@50 of 0.96. Rescored here against clients it has never seen, it lands near 0.57, and its fold to fold spread is by far the worst in the table, swinging from 0.14 to 0.96. The stratified dummy reaches a similar precision@50, so at the top of the queue my rule is not reliably beating random guessing on unseen clients. That is not a bug in either notebook. The Week 4 number was measured on all pages pooled together, where one large client dominated the top of the queue, so it was partly reporting that one client. This is the single most useful thing I learned this week, and it only showed up because the split changed.

**Learned weights on the same information do not rescue it.** `logreg_static` gets the same six static inputs my rule had access to, and it is not better in any way that matters: its precision@50 sits around the rule's and its AUC stays near a coin flip. It is steadier across folds, which counts for something, but the honest read is that the static picture of a page, how big it is and how old it is, is weak evidence about what happens next month. The problem was not that my thresholds were badly chosen. It was the information they were built from.

**Momentum is what actually moves the number.** The clean comparison is `logreg_static` against `logreg_momentum`: same model, same folds, only the trend columns added. It lifts every metric at once, precision@50 from 0.56 to 0.80 and AUC from 0.52 to 0.57. Everything moving together is the pattern you want from a feature carrying real signal rather than one fitting noise. The forest, on those same features, takes the best AUC in the table and the best or joint best precision at every K.

**The forest earns its complexity, but only on part of the evidence.** It beats the regression on AUC by a margin larger than either model's fold to fold spread. On precision@50 the gap is far smaller than the fold to fold spread, so I cannot claim it is better at the very top of the queue on this evidence. If a reviewer needed a model they could read off a coefficient table, `logreg_momentum` is a defensible choice and I would not argue hard against it.

**Honest caveat about why momentum works.** Part of it is genuine: a page already sliding in March tends to keep sliding. Part of it is arithmetic. The label compares April against the whole March total, and for a page that fell during March that total is propped up by the stronger first half, which makes the 0.8 threshold harder to clear. The check in section 4 shows the effect survives inside every volume band, so it is not merely a volume artefact, but I would want a second label definition before calling the whole effect predictive.

## 4. Errors and interpretation

Three questions: what does the model lean on, does its best signal survive a stricter look, and where is it wrong.

In [5]:
from sklearn.inspection import permutation_importance

# Permutation importance on one held out fold. Impurity importance flatters high cardinality
# columns, so I shuffle each column and measure the AUC it costs instead.
tr, te = list(gkf.split(X, y, groups))[0]
rf_i = RandomForestClassifier(n_estimators=200, min_samples_leaf=100,
                              n_jobs=-1, random_state=42).fit(X.iloc[tr], y[tr])
pi = permutation_importance(rf_i, X.iloc[te], y[te], scoring="roc_auc",
                            n_repeats=3, random_state=0, n_jobs=-1)
print("What the forest leans on (AUC lost when the column is shuffled, fold 0)")
print(pd.Series(pi.importances_mean, index=FEATS).sort_values(ascending=False).round(4).to_string())

# Does momentum survive conditioning on volume, or is it volume in disguise?
print("\nDecline rate by momentum quartile, computed inside each volume band:")
for lo, hi, nm in [(0, 50, "1-50"), (50, 200, "50-200"), (200, 1000, "200-1k"), (1000, 1e12, "1k+")]:
    s = m[(m.mar_impr > lo) & (m.mar_impr <= hi)]
    qs = s.groupby(pd.qcut(s["mom_impr"], 4, duplicates="drop", labels=False),
                   observed=True)["decline"].mean()
    print(f"  {nm:8s} n={len(s):6,d}   " + "  ".join(f"Q{i+1} {v:.3f}" for i, v in enumerate(qs.values)))

What the forest leans on (AUC lost when the column is shuffled, fold 0)
mom_impr          0.0815
ctr               0.0142
avg_position      0.0122
mar_clicks        0.0069
mom_clicks       -0.0000
h1_impr          -0.0004
h2_impr          -0.0017
days_with_impr   -0.0038
mar_impr         -0.0063
age_days         -0.0145

Decline rate by momentum quartile, computed inside each volume band:
  1-50     n=61,015   Q1 0.743  Q2 0.604  Q3 0.514  Q4 0.345
  50-200   n=31,014   Q1 0.752  Q2 0.592  Q3 0.492  Q4 0.299
  200-1k   n=39,674   Q1 0.762  Q2 0.593  Q3 0.484  Q4 0.331
  1k+      n=45,035   Q1 0.751  Q2 0.535  Q3 0.394  Q4 0.266


In [6]:
e = pd.concat(err, ignore_index=True)
e["pred"] = (e["p_rf"] >= 0.5).astype(int)

print("Where the forest is right and wrong, by March volume")
vb = pd.cut(e["mar_impr"], [0, 50, 200, 1000, 1e12], labels=["1-50", "50-200", "200-1k", "1k+"])
print(e.groupby(vb, observed=True).apply(
    lambda d: pd.Series({"n": len(d), "actual": d.y.mean(), "predicted": d.pred.mean(),
                         "accuracy": (d.y == d.pred).mean()}), include_groups=False).round(3).to_string())

fp = e[(e.pred == 1) & (e.y == 0)]
fn = e[(e.pred == 0) & (e.y == 1)]
print(f"\nfalse positives  n={len(fp):,}  median March impressions {fp.mar_impr.median():,.0f}  "
      f"median momentum {fp.mom_impr.median():.2f}")
print(f"false negatives  n={len(fn):,}  median March impressions {fn.mar_impr.median():,.0f}  "
      f"median momentum {fn.mom_impr.median():.2f}")

top = e.sort_values("p_rf", ascending=False).head(50)
print(f"\nthe forest's top 50 across folds: hit rate {top.y.mean():.3f}, "
      f"median impressions {top.mar_impr.median():,.0f}, median age {top.age_days.median():,.0f}d, "
      f"median momentum {top.mom_impr.median():.2f}")

Where the forest is right and wrong, by March volume
                n  actual  predicted  accuracy
mar_impr                                      
1-50      61015.0   0.557      0.579     0.598
50-200    31014.0   0.534      0.606     0.609
200-1k    39674.0   0.543      0.655     0.619
1k+       45035.0   0.487      0.539     0.651

false positives  n=38,928  median March impressions 180  median momentum 1.00
false negatives  n=28,567  median March impressions 109  median momentum 2.00



the forest's top 50 across folds: hit rate 0.820, median impressions 1,298, median age 259d, median momentum 0.26


### Reading the errors

**The model is close to a one signal model.** Shuffling `mom_impr` costs far more AUC than shuffling anything else, by roughly a factor of six over the next column. And `age_days`, the single input my whole Week 4 rule ranked on, comes out slightly negative, meaning shuffling it left the model no worse on this fold. My rule was ranking on the one input here that carries nothing once momentum is in the room. That is the honest verdict on my baseline, and it explains the comparison table better than anything else.

**Momentum is not volume wearing a disguise.** Inside every volume band, decline falls steadily from the lowest momentum quartile to the highest, roughly 0.75 down to 0.3. The pattern holds for pages with under fifty impressions and for pages with thousands, which is what I would expect from a real behavioural signal rather than an artefact of page size.

**Errors are concentrated in small numbers.** Accuracy is weakest in the thinnest volume band and climbs steadily with volume. This is the same weakness I named in ML-07: for a page with a handful of impressions, a twenty percent month over month move is mostly noise, so the label itself is unreliable there. The model is partly being marked wrong against a coin flip.

**The two error types have different shapes.** False positives sit at flat momentum, pages that looked steady and the model flagged anyway. False negatives sit at momentum near 2, pages clearly rising inside March that dropped in April regardless. That second group is the behaviour no decision time feature can catch: an outside event, a seasonal peak ending, or a spike that was never going to hold. The model also predicts decline more often than it happens in every volume band, so if this fed a real queue I would tune the threshold rather than leave it at 0.5.

**What I would do next**, in order: define a second label that is less fragile on thin traffic, for example requiring an impression floor before a page can be labelled at all; test whether momentum still leads once the label no longer shares the March total with it; and only then reach for gradient boosting, which I skipped because the forest already looks close to the ceiling this feature set supports.

In [7]:
# Receipts for the repo. The comparison table is the deliverable, so it is saved as run.
metrics = {
    "task": "ML-08 w05_model",
    "lane": "Refresh / Content Opportunity Scoring",
    "panel_month": "2026-03", "label_month": "2026-04",
    "n_pages": int(len(m)), "n_clients": int(m["client_hash_id"].nunique()),
    "base_decline_rate": round(float(base), 4),
    "validation": "GroupKFold(n_splits=5) grouped by client_hash_id",
    "split_honesty": {"random_split_auc": round(float(auc_r), 4),
                       "grouped_split_auc": round(float(auc_g), 4),
                       "inflation": round(float(auc_r - auc_g), 4)},
    "features_static": STATIC, "features_momentum": MOMENT,
    "excluded_as_future": ["apr_impr", "decline", "content_updated_date",
                            "last_optimized_date", "optimization_eligible_date"],
    "mean_over_folds": json.loads(mean_tbl.to_json(orient="index")),
    "std_over_folds": json.loads(std_tbl.to_json(orient="index")),
    "headline": ("rf_momentum beats the frozen ML-07 rule on the same grouped folds; "
                  "the rule's Week 4 precision@50 of 0.96 falls to roughly 0.57 once "
                  "it is scored against unseen clients"),
}
with open(os.path.join(OUT, "w05_model_metrics.json"), "w") as f:
    json.dump(metrics, f, indent=2)
print("wrote", os.path.join(OUT, "w05_model_metrics.json"))

# Guard: nothing from the label window may sit in the feature list.
FORBIDDEN = {"apr_impr", "decline", "content_updated_date", "last_optimized_date",
             "optimization_eligible_date"}
assert set(FEATS).isdisjoint(FORBIDDEN), "a future or label column reached the model"
assert abs(float(gap)) < 1e-6, "half month columns do not reconcile to the March total"
print("leakage guard passed: features are", len(FEATS), "columns, all decision time")

wrote /home/zuko/ml-internship/work/outputs/w05_model_metrics.json
leakage guard passed: features are 10 columns, all decision time


## 5. Self-check

- [x] Compared against the Week 4 baseline on the same rows, the same folds, and the same metric, with the rule rescored inside each fold rather than quoted from Week 4.
- [x] Validation design is grouped by client and justified by a measured client effect, with the random split shown alongside so the inflation is visible.
- [x] Method choice explained, simplest first, with a dummy floor and a static feature model as controls.
- [x] Useful metrics for the lane: precision at K for the queue, AUC and PR AUC for the ranking, means reported with fold spread.
- [x] Features and errors interpreted: permutation importance, momentum checked inside volume bands, error shape by volume, and both error types described.
- [x] Complexity not rewarded on its own: the forest wins on AUC, and I say plainly that its precision@50 edge sits inside the fold spread.
- [x] No future window or label derived inputs; the guard cell asserts it.
- [x] Runs top to bottom with no errors; no client names, URLs, or private queries; IDs stay pseudonymized.